# FMA Data Preprocessing

This notebook cleans the FMA metadata, emits per-subset CSVs, and converts MP3s into log-mel spectrograms saved to disk.

Two cleaned subsets are produced (controlled by `DATASET_SIZE` in `.env`):

- **small** — `subset == "small"` (≈ 8 000 tracks, 8 genres). Requires `fma_small/`. Audio existence is verified for every track.
- **medium** — `subset ∈ {small, medium}` (≈ 25 000 tracks, 16 genres). Requires `fma_small/` + `fma_metadata/`. If `fma_medium/` is also present, medium-only tracks are audio-verified and their spectrograms are extracted; otherwise they are kept in the CSV for Random Forest tabular features only.

Spectrogram formats extracted (controlled by `PREPROCESS_FOR` in `.env`):

| Format | Clip | Shape | Directory |
|--------|------|-------|-----------|
| CNN    | 10 s | (1, 64, 1001) | `spectrograms_10/` |
| CRNN   | 20 s | (1, 64, 2001) | `spectrograms_20/` |

Run `DOWNLOAD_MEDIUM=1 bash setup.sh` to download `fma_medium/` (~22 GB).

## 1. Setup

Dependencies: `torch`, `torchaudio`, `pandas`, `numpy`, `tqdm`, `soundfile`.

Configuration is read from a `.env` file in the project root (or the parent of the current
working directory). Copy `.env.example` to `.env` and set the following variables:

| Variable | Default | Description |
|---|---|---|
| `DATASET_DIR` | *(auto-detected)* | Path to the directory containing `fma_small/` and `fma_metadata/` |
| `PREPROCESSED_DIR` | `DATASET_DIR/fma_preprocessed` | Where cleaned CSVs and spectrograms are written |
| `DATASET_SIZE` | `small` | Which subset(s) to process: `small`, `medium`, or `both` |
| `PREPROCESS_FOR` | `both` | Which spectrogram formats to extract: `cnn`, `crnn`, `both`, or `none` |

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import torch
import torchaudio
import os

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable


def _load_dotenv(filename=".env"):
    """Load key=value pairs from a .env file without requiring python-dotenv."""
    for base in [Path.cwd(), Path.cwd().parent]:
        p = base / filename
        if p.exists():
            with open(p) as f:
                for line in f:
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        key, _, value = line.partition("=")
                        key = key.strip()
                        value = value.strip().strip('"').strip("'")
                        os.environ.setdefault(key, value)
            print(f"Loaded {p}")
            return
    print("No .env file found — using environment variables / built-in defaults.")

_load_dotenv()

# ── Dataset size ──────────────────────────────────────────────────────────────
_size_raw = os.environ.get("DATASET_SIZE", "small").lower().strip()
RUN_SMALL  = _size_raw in ("small", "both")
RUN_MEDIUM = _size_raw in ("medium", "both")
if not RUN_SMALL and not RUN_MEDIUM:
    raise ValueError(f"DATASET_SIZE must be 'small', 'medium', or 'both'. Got: {_size_raw!r}")

# ── Spectrogram targets ───────────────────────────────────────────────────────
# cnn  → 10 s clips → (1, 64, 1001) in spectrograms_10/
# crnn → 20 s clips → (1, 64, 2001) in spectrograms_20/
_models_raw = os.environ.get("PREPROCESS_FOR", "both").lower().strip()
if _models_raw == "none":
    RUN_CNN = RUN_CRNN = False
elif _models_raw == "both":
    RUN_CNN = RUN_CRNN = True
else:
    _targets = {t.strip() for t in _models_raw.split(",")}
    RUN_CNN  = "cnn"  in _targets
    RUN_CRNN = "crnn" in _targets

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
if "DATASET_DIR" in os.environ:
    PROJECT_CANDIDATES.insert(0, Path(os.environ["DATASET_DIR"]))

for candidate in PROJECT_CANDIDATES:
    small_audio_dir = candidate / "fma_small"
    metadata_path   = candidate / "fma_metadata" / "tracks.csv"
    if small_audio_dir.exists() and metadata_path.exists():
        PROJECT_DIR   = candidate.resolve()
        METADATA_PATH = metadata_path.resolve()
        break
else:
    raise FileNotFoundError(
        "Could not find fma_small/ and fma_metadata/tracks.csv. "
        "Set DATASET_DIR in your .env file."
    )

SMALL_AUDIO_DIR  = (PROJECT_DIR / "fma_small").resolve()
_medium_dir      = PROJECT_DIR / "fma_medium"
MEDIUM_AUDIO_DIR = _medium_dir.resolve() if _medium_dir.exists() else None

PROCESSED_DIR = (
    Path(os.environ["PREPROCESSED_DIR"]).resolve()
    if "PREPROCESSED_DIR" in os.environ
    else PROJECT_DIR / "fma_preprocessed"
)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"DATASET_DIR      : {PROJECT_DIR}")
print(f"SMALL_AUDIO_DIR  : {SMALL_AUDIO_DIR}")
print(f"MEDIUM_AUDIO_DIR : {MEDIUM_AUDIO_DIR or '(not found — medium-only audio will be skipped)'}")
print(f"METADATA_PATH    : {METADATA_PATH}")
print(f"PREPROCESSED_DIR : {PROCESSED_DIR}")
print(f"device           : {device}")
print(f"RUN_SMALL        : {RUN_SMALL}  (DATASET_SIZE={_size_raw!r})")
print(f"RUN_MEDIUM       : {RUN_MEDIUM}")
print(f"RUN_CNN          : {RUN_CNN}   (PREPROCESS_FOR={_models_raw!r})")
print(f"RUN_CRNN         : {RUN_CRNN}")

## 2. Load `tracks.csv`

`tracks.csv` uses a two-row header, so columns load as a `MultiIndex` like `("track", "genre_top")`. This stage selects the required fields and flattens them into a normal DataFrame.

In [ ]:
def track_id_to_audio_path(track_id: int, subset: str = "small") -> Path:
    """Return the expected MP3 path for a track.

    Tracks with subset=='small' live in fma_small/.
    Tracks with subset=='medium' live in fma_medium/ when it is available,
    otherwise fma_small/ is used as a fallback (the file will be missing and
    the audio-existence check will drop the track during cleaning).
    """
    filename = f"{int(track_id):06d}.mp3"
    rel = Path(filename[:3]) / filename
    if subset == "medium" and MEDIUM_AUDIO_DIR is not None:
        return MEDIUM_AUDIO_DIR / rel
    return SMALL_AUDIO_DIR / rel


tracks = pd.read_csv(METADATA_PATH, index_col=0, header=[0, 1])

# The track_id index is reset so metadata_all stores it only as a column.
tracks_flat = tracks.reset_index(drop=True)

metadata_all = pd.DataFrame({
    "track_id": tracks.index.astype(int).to_numpy(),
    "subset":   tracks_flat[("set", "subset")].to_numpy(),
    "split":    tracks_flat[("set", "split")].to_numpy(),
    "genre":    tracks_flat[("track", "genre_top")].to_numpy(),
    "duration": tracks_flat[("track", "duration")].to_numpy(),
    "bit_rate": tracks_flat[("track", "bit_rate")].to_numpy(),
    "title":    tracks_flat[("track", "title")].to_numpy(),
    "artist":   tracks_flat[("artist", "name")].to_numpy(),
})
metadata_all["audio_path"] = metadata_all.apply(
    lambda row: track_id_to_audio_path(row["track_id"], row["subset"]), axis=1
)

print(f"Total tracks in tracks.csv: {len(metadata_all):,}")
print(metadata_all["subset"].value_counts())
metadata_all.head()

## 3. Clean each subset

Each subset drops tracks with a null genre, tracks in the `FAILED` list (known broken audio),
and tracks whose MP3 is missing or empty on disk.

Which subsets are cleaned depends on `DATASET_SIZE` in `.env`:
- **small** — `subset == "small"` (≈ 8 000 tracks, 8 genres). Audio-file existence is verified for every track.
- **medium** — `subset ∈ {small, medium}` (≈ 25 000 tracks, 16 genres). Audio checks only run for the small portion; medium-only rows are kept for Random Forest tabular features even without local audio files.

In [ ]:
def keep(mask, df, label):
    old = len(df)
    df = df[mask]
    print(f"{label:42s} {old - len(df):>5d} dropped, {len(df):>5d} left")
    return df


def is_file_nonempty(path: Path) -> bool:
    try:
        return path.exists() and path.stat().st_size > 0
    except OSError:
        return False


FAILED = [
    1440, 26436, 28106, 29166, 29167, 29168, 29169, 29170, 29171, 29172,
    29173, 29179, 38903, 43903, 56757, 57603, 59361, 62095, 62954, 62956,
    62957, 62959, 62971, 75461, 80015, 86079, 92345, 92346, 92347, 92348,
    92349, 92350, 92351, 92352, 92353, 92354, 92355, 92356, 92357, 92358,
    92359, 92360, 92361, 96426, 104623, 106719, 109714, 114448, 114501, 114528,
    115235, 117759, 118003, 118004, 127827, 130296, 130298, 131076, 135804, 136486,
    144769, 144770, 144771, 144773, 144774, 144775, 144776, 144777, 144778, 152204,
    154923,
]

metadata = None
genre_to_idx_small = None

if RUN_SMALL:
    print("=== Cleaning small ===")
    metadata = metadata_all.copy()
    metadata = keep(metadata["subset"] == "small", metadata, "subset == small")
    metadata = keep(metadata["genre"].notna(), metadata, "genre not null")
    metadata = keep(metadata["audio_path"].map(is_file_nonempty), metadata, "audio file present and non-empty")
    metadata = keep(~metadata["track_id"].isin(FAILED), metadata, "not in FAILED list")

    metadata = metadata.sort_values("track_id").reset_index(drop=True)
    genres_small_sorted = sorted(metadata["genre"].unique())
    genre_to_idx_small = {g: i for i, g in enumerate(genres_small_sorted)}
    metadata["label"] = metadata["genre"].map(genre_to_idx_small).astype(int)

    print("\nSmall genre distribution:")
    print(metadata["genre"].value_counts().sort_index())
    print("\nSmall split distribution:")
    print(metadata["split"].value_counts())
else:
    print("Skipping small subset (DATASET_SIZE={!r}).".format(_size_raw))

In [ ]:
metadata_medium = None
genre_to_idx_medium = None

if RUN_MEDIUM:
    print("=== Cleaning medium ===")
    metadata_medium = metadata_all.copy()

    metadata_medium = keep(
        metadata_medium["subset"].isin(["small", "medium"]),
        metadata_medium,
        "subset in {small, medium}",
    )
    metadata_medium = keep(metadata_medium["genre"].notna(), metadata_medium, "genre not null")
    metadata_medium = keep(~metadata_medium["track_id"].isin(FAILED), metadata_medium, "not in FAILED list")

    is_small_track = metadata_medium["subset"] == "small"
    has_audio      = metadata_medium["audio_path"].map(is_file_nonempty)

    if MEDIUM_AUDIO_DIR is not None:
        # fma_medium/ is present — require audio for all tracks.
        metadata_medium = keep(has_audio, metadata_medium, "audio file present and non-empty")
    else:
        # fma_medium/ absent — small tracks must have audio; medium-only tracks are
        # kept anyway so Random Forest can still use their tabular features.
        metadata_medium = keep(
            has_audio | (~is_small_track),
            metadata_medium,
            "small-portion audio present (medium-only kept for RF)",
        )

    metadata_medium = metadata_medium.sort_values("track_id").reset_index(drop=True)
    genres_medium_sorted = sorted(metadata_medium["genre"].unique())
    genre_to_idx_medium = {g: i for i, g in enumerate(genres_medium_sorted)}
    metadata_medium["label"] = metadata_medium["genre"].map(genre_to_idx_medium).astype(int)

    print("\nMedium genre distribution:")
    print(metadata_medium["genre"].value_counts().sort_index())
    print("\nMedium split distribution:")
    print(metadata_medium["split"].value_counts())
else:
    print("Skipping medium subset (DATASET_SIZE={!r}).".format(_size_raw))

## 4. Save the cleaned CSVs

Each subset produces:
- `tracks_clean_{subset}.csv` — the full cleaned frame.
- `tracks_clean_{subset}_{split}.csv` — per-split slices for model training notebooks.
- `genre_to_idx_{subset}.csv` — the genre → label mapping.

In [ ]:
def save_clean_csvs(frame: pd.DataFrame, subset_name: str, genre_to_idx: dict) -> None:
    base_path = PROCESSED_DIR / f"tracks_clean_{subset_name}.csv"
    frame.to_csv(base_path, index=False)
    print(f"Wrote {base_path} ({len(frame):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = frame[frame["split"] == split_name]
        split_path = PROCESSED_DIR / f"tracks_clean_{subset_name}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    genre_map_path = PROCESSED_DIR / f"genre_to_idx_{subset_name}.csv"
    pd.DataFrame(
        sorted(genre_to_idx.items(), key=lambda kv: kv[1]),
        columns=["genre", "label"],
    ).to_csv(genre_map_path, index=False)
    print(f"Wrote {genre_map_path}")


if RUN_SMALL:
    save_clean_csvs(metadata, "small", genre_to_idx_small)

if RUN_MEDIUM:
    print()
    save_clean_csvs(metadata_medium, "medium", genre_to_idx_medium)

## 5. Spectrogram extraction

CNN and CRNN use different clip lengths, which produce different spectrogram shapes:

| Model | Clip length | Output shape | Output directory |
|-------|-------------|--------------|------------------|
| CNN   | 10 s        | (1, 64, 1001) | `spectrograms_10/` |
| CRNN  | 20 s        | (1, 64, 2001) | `spectrograms_20/` |

Common settings: 16 kHz mono, `n_fft=400`, `hop_length=160`, 64 mel bins, log power scale,
per-clip z-score normalisation, saved as `float32` `.npy` files.

Which sections run is controlled by `PREPROCESS_FOR` in `.env`.

In [ ]:
SAMPLE_RATE = 16_000
N_FFT       = 400
HOP_LENGTH  = 160
N_MELS      = 64


def _make_transforms(clip_seconds: int):
    num_samples = SAMPLE_RATE * clip_seconds
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
    ).to(device)
    to_db = torchaudio.transforms.AmplitudeToDB(stype="power").to(device)
    return mel, to_db, num_samples


def load_clip(path: Path, num_samples: int) -> torch.Tensor:
    audio, sr = sf.read(path, dtype="float32", always_2d=True)
    waveform = torch.from_numpy(audio.T).mean(dim=0, keepdim=True)
    if sr != SAMPLE_RATE:
        waveform = torchaudio.functional.resample(waveform, sr, SAMPLE_RATE)
    if waveform.shape[1] > num_samples:
        start = (waveform.shape[1] - num_samples) // 2
        waveform = waveform[:, start:start + num_samples]
    elif waveform.shape[1] < num_samples:
        waveform = torch.nn.functional.pad(waveform, (0, num_samples - waveform.shape[1]))
    return torch.nan_to_num(waveform, nan=0.0, posinf=0.0, neginf=0.0).clamp(-1.0, 1.0)


@torch.no_grad()
def waveform_to_spec(waveform: torch.Tensor, mel_transform, to_db) -> torch.Tensor:
    spec = to_db(mel_transform(waveform.to(device)))
    spec = torch.nan_to_num(spec, nan=0.0, posinf=0.0, neginf=0.0)
    mean = spec.mean(dim=(-2, -1), keepdim=True)
    std  = spec.std(dim=(-2, -1), keepdim=True).clamp_min(1e-6)
    return torch.nan_to_num((spec - mean) / std, nan=0.0, posinf=0.0, neginf=0.0).cpu()


def extract_spectrograms(
    metadata_df: pd.DataFrame,
    clip_seconds: int,
    spec_dir: Path,
    manifest_prefix: str,
    skip_existing: bool = True,
) -> pd.DataFrame:
    """Extract mel spectrograms for every row in metadata_df and save manifests.

    Returns the manifest DataFrame (one row per successfully extracted spectrogram).
    """
    mel, to_db, num_samples = _make_transforms(clip_seconds)
    spec_dir.mkdir(parents=True, exist_ok=True)

    def _spec_path(track_id: int) -> Path:
        fname = f"{int(track_id):06d}.npy"
        return spec_dir / fname[:3] / fname

    spec_paths, dropped_rows = [], []
    n_skipped = n_written = 0

    for row in tqdm(metadata_df.itertuples(index=False), total=len(metadata_df),
                    desc=f"Extracting {manifest_prefix}"):
        out_path = _spec_path(row.track_id)
        out_path.parent.mkdir(parents=True, exist_ok=True)

        if skip_existing and out_path.exists():
            spec_paths.append(str(out_path))
            n_skipped += 1
            continue

        try:
            waveform = load_clip(Path(row.audio_path), num_samples)
            spec = waveform_to_spec(waveform, mel, to_db).numpy().astype(np.float32)
        except Exception as exc:
            dropped_rows.append({
                "track_id": row.track_id,
                "audio_path": str(row.audio_path),
                "error": f"{type(exc).__name__}: {exc}",
            })
            spec_paths.append(None)
            continue

        np.save(out_path, spec)
        spec_paths.append(str(out_path))
        n_written += 1

    print(f"\nWrote {n_written:,} new, reused {n_skipped:,} existing, dropped {len(dropped_rows):,} on error.")
    if dropped_rows:
        print(pd.DataFrame(dropped_rows).to_string(index=False))

    manifest_df = metadata_df.copy()
    manifest_df["spectrogram_path"] = spec_paths
    manifest_df = manifest_df[manifest_df["spectrogram_path"].notna()].reset_index(drop=True)
    cols = ["track_id", "split", "genre", "label", "spectrogram_path",
            "audio_path", "duration", "bit_rate", "title", "artist"]
    manifest_df = manifest_df[cols]

    manifest_path = PROCESSED_DIR / f"{manifest_prefix}_manifest.csv"
    manifest_df.to_csv(manifest_path, index=False)
    print(f"Wrote {manifest_path} ({len(manifest_df):,} rows)")

    for split_name in ("training", "validation", "test"):
        split_df = manifest_df[manifest_df["split"] == split_name]
        split_path = PROCESSED_DIR / f"{manifest_prefix}_{split_name}.csv"
        split_df.to_csv(split_path, index=False)
        print(f"Wrote {split_path} ({len(split_df):,} rows)")

    if dropped_rows:
        err_path = PROCESSED_DIR / f"{manifest_prefix}_extraction_errors.csv"
        pd.DataFrame(dropped_rows).to_csv(err_path, index=False)
        print(f"Wrote {err_path} ({len(dropped_rows):,} rows)")

    return manifest_df


print("Spectrogram helpers defined.")

## 6. CNN spectrogram extraction (10 s clips → `spectrograms_10/`)

Runs when `PREPROCESS_FOR` is `cnn` or `both`. Extracts from the subset(s) selected by `DATASET_SIZE`.

In [ ]:
manifest_cnn_small  = None
manifest_cnn_medium = None

if RUN_CNN:
    cnn_spec_dir = PROCESSED_DIR / "spectrograms_10"

    if RUN_SMALL:
        print("=== CNN · small (10 s clips) ===")
        manifest_cnn_small = extract_spectrograms(
            metadata,
            clip_seconds=10,
            spec_dir=cnn_spec_dir / "small",
            manifest_prefix="spectrograms_10_small",
        )

    if RUN_MEDIUM:
        print("\n=== CNN · medium (10 s clips) ===")
        manifest_cnn_medium = extract_spectrograms(
            metadata_medium,
            clip_seconds=10,
            spec_dir=cnn_spec_dir / "medium",
            manifest_prefix="spectrograms_10_medium",
        )
else:
    print("Skipping CNN spectrogram extraction (PREPROCESS_FOR={!r}).".format(_models_raw))

## 7. CRNN spectrogram extraction (20 s clips → `spectrograms_20/`)

Runs when `PREPROCESS_FOR` is `crnn` or `both`. Extracts from the subset(s) selected by `DATASET_SIZE`.

In [ ]:
manifest_crnn_small  = None
manifest_crnn_medium = None

if RUN_CRNN:
    crnn_spec_dir = PROCESSED_DIR / "spectrograms_20"

    if RUN_SMALL:
        print("=== CRNN · small (20 s clips) ===")
        manifest_crnn_small = extract_spectrograms(
            metadata,
            clip_seconds=20,
            spec_dir=crnn_spec_dir / "small",
            manifest_prefix="spectrograms_20_small",
        )

    if RUN_MEDIUM:
        print("\n=== CRNN · medium (20 s clips) ===")
        manifest_crnn_medium = extract_spectrograms(
            metadata_medium,
            clip_seconds=20,
            spec_dir=crnn_spec_dir / "medium",
            manifest_prefix="spectrograms_20_medium",
        )
else:
    print("Skipping CRNN spectrogram extraction (PREPROCESS_FOR={!r}).".format(_models_raw))

## 8. Sanity check

Loads the first spectrogram from each generated manifest and prints shape, dtype, and value statistics.

In [ ]:
checks = [
    ("CNN   · small",  manifest_cnn_small),
    ("CNN   · medium", manifest_cnn_medium),
    ("CRNN  · small",  manifest_crnn_small),
    ("CRNN  · medium", manifest_crnn_medium),
]

any_checked = False
for label, mf in checks:
    if mf is None or len(mf) == 0:
        continue
    any_checked = True
    first = mf.iloc[0]
    loaded = np.load(first["spectrogram_path"])
    print(f"[{label}]  track {first['track_id']} ({first['genre']}, {first['split']})")
    print(f"  shape={loaded.shape}  dtype={loaded.dtype}  "
          f"mean={loaded.mean():.4f}  std={loaded.std():.4f}  "
          f"min={loaded.min():.2f}  max={loaded.max():.2f}")

if not any_checked:
    print("No spectrograms were extracted (PREPROCESS_FOR='none' or no subsets selected).")